In [1]:
import numpy as np
from klotho import play, set_audio_engine
from klotho.thetos import CompositionalUnit as UC, ToneInstrument as JsInst
from klotho.chronos import TemporalBlock as BT
from klotho.tonos import Scale
from klotho.topos.collections.sequences import Pattern

set_audio_engine("tone")

### PREVIEW:

#### Want to make rhythms like this?

In [2]:
# Chronostasis
np.random.seed(0)
tempus = '10/16'
beat = '1/16'
bpm = 140
n_bars = 4
S1 = ((3, (1,)*4), (4, (1,)*6), (3, (1,)*4))
S2 = ((5, (1,)*5),)*2
inst_pat_1 = Pattern([JsInst.HatClosed(), [JsInst.HatClosed(), [[JsInst.TomHigh(), JsInst.TomMid()], [JsInst.HatOpen(), [JsInst.Ride(decay=0.2), JsInst.HatClosed()]]]]])
uc1 = UC(tempus=tempus, prolatio=S1, beat=beat, bpm=bpm)
uts1 = uc1.repeat(n_bars)
for unit in uts1:
    unit.leaves.set_instrument(inst_pat_1)
    unit.leaves.set_pfields(vel=lambda: np.random.uniform(0.001, 0.25))
inst_pat_2 = Pattern([[[JsInst.Kick("kick_punchy", punch=16), JsInst.Kick("kick_pitchy", pitchDecay=0.05, decay=0.9)], JsInst.Snare()], JsInst.Kick()])
uc2 = UC(tempus=tempus, prolatio=S2, beat=beat, bpm=bpm)
uts2 = uc2.repeat(n_bars)
for unit in uts2:
    unit.leaves.set_instrument(inst_pat_2)
    unit.leaves.set_pfields(vel=lambda: np.random.uniform(0.75, 0.9))
play(BT([uts1, uts2]))

...or like this?

In [3]:
# Entertain Me
np.random.seed(1)
tempus = '36/16'
beat = '1/8'
bpm = 184
n_bars = 2
scale = Scale.phrygian().root('B3')
S1 = ((20, ((5, (1,)*5),)*4), (15, ((3, (1,)*3),)*5))
uc_mel = UC(tempus=tempus, prolatio=S1, beat=beat, bpm=bpm, inst=JsInst.Kalimba())
limbs = uc_mel.at_depth(1)
L0, L1 = limbs[0].id, limbs[-1].id
limbs[0].set_mfields(idx=0, direction=1, offset=0)
limbs[1].set_mfields(idx=len(scale), direction=-1, offset=0)
uc_mel.successors(L1).set_mfields(offset=lambda c: c.total - c.index)
for branch in uc_mel.at_depth(2):
    uc_mel.leaves_of(branch).set_pfields(
        freq=lambda c: scale[c.mfields['offset'] + c.mfields['idx'] + c.mfields['direction'] * c.index].freq
    )
uc_ds = UC(tempus=tempus, prolatio=S1, beat=beat, bpm=bpm)
uc_ds.leaves_of(L0).set_instrument(Pattern([[JsInst.Kick(), JsInst.Snare()], JsInst.HatClosed()]))
uc_ds.leaves_of(L1).set_instrument(lambda c: JsInst.HatOpen(vel=0.1) if c.index % 2 == 0 else JsInst.HatClosed())
uc_bs = UC(tempus=tempus, prolatio=S1, beat=beat, bpm=bpm, inst=JsInst.Bassy(freq=scale[-len(scale)*2].freq, vel=0.2))
bs_pat = Pattern([0, 0, [1, [3, [4, -3]]]])
uc_bs.make_rest(limbs[-1].id)
seq_bs = uc_bs.repeat(n_bars)
for unit in seq_bs:
    unit.leaves.set_pfields(freq=lambda: scale[next(bs_pat) - len(scale)*2].freq)
play(BT([uc_mel.repeat(n_bars), uc_ds.repeat(n_bars), seq_bs]))

...or like this?

In [4]:
# Polyriddim
np.random.seed(2)
S1 = ((1, ((6, (1,)*7), (8, (1,)*11))), (1, ((6, ((3, (1,)*4), 1, (2, (1,)*3))), (8, ((3, (1,)*4), (3, (1,)*4), (5, (1,)*5))))), (1, ((6, (2, (3, (1,)*4), (2, (1,)*4))), (8, ((2, (1,)*3), (2, (1,)*4), (2, (1,)*5), (2, (1,)*5))))), (1, ((6, ((2, (1,)*3), (2, (1,)*3), (2, (1,)*3))), (8, (5, (6, (1,)*11))))))
S2 = ((7, ((3, (1,)*3), (4, (1,)*4))),)*4
tempus = '28/16'
beat = '1/16'
bpm = 122.5
inst1_pat = Pattern([
    [JsInst.Kick(), [
        [JsInst.Kick("kick_punch", punch=9, click=0.6), JsInst.Snare("snare_body", body=0.8)], JsInst.TomMid()]],
    [JsInst.Kick("kick_click", click=0.8), [
        [JsInst.Snare(), JsInst.TomHigh(punch=8, decay=0.9)], JsInst.TomLow()]]
])
n_bars = 2
uc1 = UC(tempus=tempus, prolatio=S1, beat=beat, bpm=bpm)
uts1 = uc1.repeat(n_bars)
for unit in uts1:
    unit.sparsify(0.33)
    unit.leaves.set_instrument(inst1_pat)
    unit.leaves.set_pfields(vel=lambda: np.random.uniform(0.25, 0.85))
uc2 = UC(tempus=tempus, prolatio=S2, beat=beat, bpm=bpm)
uts2 = uc2.repeat(n_bars)
for unit in uts2:
    for branch in unit.at_depth(2):
        branch.set_instrument(JsInst.HatClosed())
        unit.successors(branch)[-1].set_instrument(JsInst.HatOpen())
        unit.leaves_of(branch).set_pfields(vel=lambda: np.random.uniform(0.05, 0.25))
scale = Scale.locrian().root('Eb2')
scl_pat = Pattern([[0, -1, [0, -3]], [1, [3, 4]]])
uc3 = UC(tempus=tempus, prolatio=S1, beat=beat, bpm=bpm, inst=JsInst.Bassy())
uts3 = uc3.repeat(n_bars)
for unit in uts3:
    unit.sparsify(0.67)
    unit.leaves.set_pfields(freq=lambda: scale[next(scl_pat)].freq, vel=lambda: np.random.uniform(0.1, 0.5))
play(BT([uts1, uts2, uts3]))

#### Yes? Then keep reading!

#### No? Well... keep reading anyway!